# NB02 — Estimation: Primary OLS + verdict-tree mapping

**Spec:** `2026-04-27-simple-beta-pair-d-design.md` v1.3.1, sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`.

**Spec sections governing this notebook:** §3.1 (PASS/FAIL/ESCALATE/SUBSTRATE_TOO_NOISY decision tree), §3.3 (Clause A ESCALATE), §3.4 (Clause B / B-ii), §3.5 (SUBSTRATE_TOO_NOISY override), §5.1 (logit transform), §5.2 (X panel), §5.3 (composite β contrast), §5.4 (lag set), §5.5 (HAC bandwidth rule-of-thumb), §8.1 (joint verdict integration), §9.1/§9.2/§9.7 (anti-fishing governance).

**Reproducibility target:** byte-identical reproduction of `results/primary_ols.json` (sha256 `d4790e743cdec62f1368cab1833e1266cb2da763d7c0931dd732bdf3d17938cf`) — round-trip asserted in §3.

**Authoring discipline:** trio-checkpoint per memory `feedback_notebook_trio_checkpoint`. Three trios: load + descriptive baseline (handed off from NB01); primary OLS estimate; verdict-tree mapping. HALT after each trio.

---

## §1 Re-load panel + handoff verification from NB01

Re-load `panel_combined.parquet`; verify sha256 matches `env.PANEL_SHA256`; verify N=134 invariant from NB01 fingerprint.

In [1]:
"""§1 — NB01 handoff verify (single-cell fold; cryptographic gate before estimation).

Per spec [@abrigoSimpleBetaSpec2026] §4 (window 2015-01-31 → 2026-02-28, N = 134
post-CORRECTIONS-α' Option-α') and §9.1 / §9.2 / §9.7 (sha256 governance), NB02
cannot start estimating until it has cryptographically verified that it is
operating on the same panel state NB01 validated. Re-loads NB01 fingerprint
JSON if present; falls back to inline panel sanity check if NB01 has not yet
executed (keeps NB02 independently runnable for orchestrator dispatch). Spec
sha256 pinned at 964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659
(env.SPEC_SHA256). Pattern source: contracts/notebooks/fx_vol_cpi_surprise/
Colombia/ closed FAIL pipeline (2026-04-19). PASS here unlocks §2 primary OLS.
"""
import hashlib
import json

import numpy as np
import pandas as pd

import env

env.pin_seed(env.SEED)

# ── Re-verify panel sha256 (cryptographic handoff from NB01) ───────────────
panel_sha_observed = hashlib.sha256(env.PANEL_PARQUET.read_bytes()).hexdigest()
assert panel_sha_observed == env.PANEL_SHA256, (
    f"Panel sha256 drift: observed {panel_sha_observed!r}, pinned {env.PANEL_SHA256!r}."
)

# ── NB01 fingerprint handoff (graceful gate — inline sanity if missing) ────
if env.PANEL_FINGERPRINT_PATH.exists():
    fp = json.loads(env.PANEL_FINGERPRINT_PATH.read_text())
    assert fp["panel_sha256"] == env.PANEL_SHA256, (
        f"NB01 fingerprint panel_sha256 drift: {fp['panel_sha256']!r} vs {env.PANEL_SHA256!r}."
    )
    assert fp["spec_sha256"] == env.SPEC_SHA256, (
        f"NB01 fingerprint spec_sha256 drift: {fp['spec_sha256']!r} vs {env.SPEC_SHA256!r}."
    )
    assert fp["n"] == env.N_EXPECTED, (
        f"NB01 fingerprint N drift: {fp['n']} vs {env.N_EXPECTED}."
    )
    handoff_source = "nb1_panel_fingerprint.json"
else:
    handoff_source = "inline (NB01 not yet executed)"

# ── Inline panel sanity (always runs as defense-in-depth) ──────────────────
df = pd.read_parquet(env.PANEL_PARQUET).sort_values("timestamp_utc").reset_index(drop=True)
assert len(df) == env.N_EXPECTED, f"N invariant violated: {len(df)} != {env.N_EXPECTED}."
ts = pd.to_datetime(df["timestamp_utc"], utc=True)
assert ts.iloc[0].strftime("%Y-%m-%d") == env.WINDOW_START
assert ts.iloc[-1].strftime("%Y-%m-%d") == env.WINDOW_END

# Required columns + zero-NaN gate (spec §5.1 + §5.2)
required_cols = [
    "timestamp_utc", "Y_broad_logit", "Y_broad_raw",
    "log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12",
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
assert int(df[required_cols].isna().sum().sum()) == 0, "NaN in required columns; HALT."

print(f"§1 handoff source: {handoff_source}")
print(f"Panel sha256 OK: {env.PANEL_SHA256}")
print(f"N = {len(df)} (window {env.WINDOW_START} → {env.WINDOW_END})")
print("§1 handoff verify PASS. NB02 §2 may proceed.")

§1 handoff source: nb1_panel_fingerprint.json
Panel sha256 OK: 6d7d9e60dad1715ce86e8adb7b3d44ba236d0b063796293b40575994a9363edf
N = 134 (window 2015-01-31 → 2026-02-28)
§1 handoff verify PASS. NB02 §2 may proceed.


## §2 Primary OLS — spec-verbatim per §5.3

Trio 2 (numbered against the orchestrator's per-NB trio count): primary OLS of `Y_broad_logit` on `[const, log_cop_usd_lag6, log_cop_usd_lag9, log_cop_usd_lag12]` with HAC SE (Newey-West, bandwidth = `floor(4*(N/100)^(2/9))` = 4 for N=134). Composite β contrast `c = [0, 1, 1, 1]` (intercept-excluded); composite SE via `sqrt(c · Σ_HAC · c)`. One-sided p per spec §5.3 sign-correct semantics.

**Trio 2 — why-block (4-part citation block per `feedback_notebook_citation_block`)**

1. **Reference.** Spec [@abrigoSimpleBetaSpec2026] sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`. Sections governing this trio:
   - **§5.1 (Y construction)** — `Y_broad_logit = log(Y_broad / (1 − Y_broad))` is the spec-pinned outcome variable. Y_broad is the Colombian young-worker (14–28) services-sector employment share over CIIU Rev.4 sections G–T per DANE GEIH micro-data; the logit transform handles the (0, 1) bounded-share support and is licensed because Y is observed strictly interior to (0, 1) at every row (NB01 §1 PASS).
   - **§5.2 (X construction)** — Three lagged regressors `log_cop_usd_lag{6, 9, 12}`: monthly end-of-period nominal COP/USD spot from Banco de la República TRM [@banrepCOPUSD], natural-log-transformed and lagged by `k ∈ {6, 9, 12}` months.
   - **§5.3 (primary specification + composite-β contrast)** — `Y_t (logit) = α + β_6·X_{t-6} + β_9·X_{t-9} + β_12·X_{t-12} + ε_t`; the test statistic is `β_composite = β_6 + β_9 + β_12` with variance `Var(β_composite) = c'Σc` where `c = (0, 1, 1, 1)` (intercept-excluded contrast); HAC-Newey-West robust covariance per [@neweyWest1987simple]; one-sided `H0: β_composite ≤ 0` vs `H1: β_composite > 0` at α = 0.05.
   - **§5.5 (HAC bandwidth rule-of-thumb)** — `bw = ⌊4·(N/100)^(2/9)⌋` per [@andrews1991heteroskedasticity]; for N = 134 this evaluates to 4.
   - One-sided p-value semantics: sign-correct under asymptotic-normal approximation — `p_one = 1 − Φ(t)` if `t > 0`, else `p_one = Φ(t)`.

2. **Why used (the spec-pinned primary).** This is the load-bearing test of the entire Pair D Stage-1 hypothesis. The composite-β contrast is the *only* test the spec pre-registers as primary; individual lag-β t-statistics are documented but NOT load-bearing because (a) the three lagged COP/USD series are highly serially correlated (the lag structure shifts the same monthly TRM across overlapping windows), so individual-lag SEs are inflated by the regressor cross-correlation, and (b) per spec §5.3 + MQS R2, the negative cross-coefficient covariance terms in `Σ_HAC` *deflate* the composite SE relative to the naive `sum(SE_i²)^{1/2}`, so the composite test is more powerful than the individual-lag tests against the same alternative. Reading "individual lags weren't significant" as "the hypothesis fails" is methodologically incorrect under the spec [@andrews1991heteroskedasticity; @neweyWest1987simple]. The spec choice of the composite contrast is itself the anti-fishing pre-commitment: the contrast vector is fixed before estimation and may not be re-tuned.

3. **Relevance to results.** This trio produces every numerical input to the §3 verdict-tree mapping. Specifically: `β_composite_point`, `se_hac`, `t_stat`, `p_one_sided`, `ci95_lower_one_sided`, individual-lag sign pattern, and the four §3.4-relevant residual diagnostics (skew, excess kurtosis, ADF on residuals, Ljung-Box at lags 4 and 12). Round-trip assertion against the committed `primary_ols.json` (sha256 `d4790e743cdec62f1368cab1833e1266cb2da763d7c0931dd732bdf3d17938cf` per env.PRIMARY_OLS_SHA256) provides the byte-deterministic reproduction proof for Option β.

4. **Connection to simulator (Stage-2 M-sketch).** The composite-β point estimate and its HAC SE quantify the Y-on-X transmission-strength magnitude that an ideal-scenario Panoptic perpetual on the Colombian-structural-transformation index would settle against. A larger composite-β → larger expected one-period Y movement per log-COP/USD shock → larger gamma exposure for a Panoptic position parameterized at the historical X support. The lag-pattern (6 / 9 / 12) further informs the time-horizon dimensioning of the Panoptic position (theta-roll cadence). Specifically, the ratio `β_6 / β_composite` measures how front-loaded the offshoring-contracting cycle is at the 6-month horizon vs. the 9–12-month horizon — a policy-relevant input to the M-sketch's roll-frequency selection.

In [2]:
import hashlib
import json

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.tsa.stattools import adfuller

import env

env.pin_seed(env.SEED)

# ── Load panel (deterministic ordering matches Phase-2 script) ─────────────
df = pd.read_parquet(env.PANEL_PARQUET).sort_values("timestamp_utc").reset_index(drop=True)
n = len(df)
assert n == env.N_EXPECTED, f"N drift: {n} != {env.N_EXPECTED}"

# ── Sanity: HAC bandwidth rule-of-thumb ⌊4·(N/100)^(2/9)⌋ ──────────────────
bw = int(np.floor(4.0 * (n / 100.0) ** (2.0 / 9.0)))
assert bw == env.NEWEY_WEST_BANDWIDTH_PRIMARY, (
    f"HAC bandwidth drift: computed {bw}, pinned {env.NEWEY_WEST_BANDWIDTH_PRIMARY}"
)

# ── Build design matrix (spec §5.3 verbatim — NO methodology dummy) ────────
y = df["Y_broad_logit"].astype(float).to_numpy()
X_lags = df[["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"]].astype(float).to_numpy()
X_design = sm.add_constant(X_lags, prepend=True)  # column order: [const, lag6, lag9, lag12]
column_names_primary = ["const", "log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"]

# ── Fit primary OLS with HAC-NW(4) covariance ──────────────────────────────
fit = sm.OLS(y, X_design).fit(cov_type="HAC", cov_kwds={"maxlags": bw})

# ── Composite-β contrast (intercept-excluded, c = [0, 1, 1, 1]) ────────────
c_primary = np.array([0.0, 1.0, 1.0, 1.0])
beta_hat = np.asarray(fit.params)
sigma_hac = np.asarray(fit.cov_params())
beta_comp_point = float(c_primary @ beta_hat)
var_comp = float(c_primary @ sigma_hac @ c_primary)
assert var_comp > 0, f"Composite-β variance {var_comp} ≤ 0 — numerical pathology"
se_comp = float(np.sqrt(var_comp))
t_comp = beta_comp_point / se_comp

# Sign-correct one-sided p (asymptotic normal under §5.3 H1: β > 0)
if t_comp > 0:
    p_one_sided = float(1.0 - stats.norm.cdf(t_comp))
else:
    p_one_sided = float(stats.norm.cdf(t_comp))
p_two_sided = float(2.0 * (1.0 - stats.norm.cdf(abs(t_comp))))

# One-sided 95% CI lower bound
z_95 = float(stats.norm.ppf(0.95))  # ≈ 1.6448536269514722
ci95_lower_one_sided = beta_comp_point - z_95 * se_comp
z_975 = float(stats.norm.ppf(0.975))
ci_two_lower = beta_comp_point - z_975 * se_comp
ci_two_upper = beta_comp_point + z_975 * se_comp

# ── Individual-lag sign pattern (informational; NOT load-bearing per §5.3) ──
individual = {}
for i, col in enumerate(["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"], start=1):
    b = float(fit.params[i])
    se_i = float(fit.bse[i])
    t_i = b / se_i if se_i > 0 else float("nan")
    p_two_i = float(2.0 * (1.0 - stats.norm.cdf(abs(t_i))))
    sign = "+" if b > 0 else ("-" if b < 0 else "0")
    individual[col] = {"point": b, "se_hac": se_i, "t": t_i, "p_two_sided": p_two_i, "sign": sign}
sign_pattern = "/".join(individual[c]["sign"] for c in
                        ["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"])

# ── Joint F-test (β_6 = β_9 = β_12 = 0) ─────────────────────────────────────
R_joint = np.array([[0., 1., 0., 0.], [0., 0., 1., 0.], [0., 0., 0., 1.]])
f_test = fit.f_test(R_joint)

# ── Residual diagnostics (spec §3.4 B-ii relevance: skew, excess kurtosis) ─
resid = np.asarray(fit.resid)
bp_lm, bp_p, _, _ = het_breuschpagan(resid, X_design)
dw = float(durbin_watson(resid))
jb_stat, jb_p, skew_resid, kurt_resid_raw = jarque_bera(resid)
excess_kurt = float(kurt_resid_raw) - 3.0
adf_stat, adf_p, adf_lag, adf_n, adf_crit, _ = adfuller(resid, autolag="AIC")
lb = acorr_ljungbox(resid, lags=[4, 12], return_df=True)

# ── Round-trip assertion: committed JSON sha256 + numerical reconciliation ─
committed_json_sha = hashlib.sha256(env.PRIMARY_OLS_JSON.read_bytes()).hexdigest()
assert committed_json_sha == env.PRIMARY_OLS_SHA256, (
    f"primary_ols.json sha256 drift: observed {committed_json_sha!r}, "
    f"pinned {env.PRIMARY_OLS_SHA256!r}."
)
committed = json.loads(env.PRIMARY_OLS_JSON.read_text())
prim = committed["primary_spec_verbatim"]
assert np.isclose(beta_comp_point, prim["composite_beta"]["point"], atol=1e-9), (
    f"β_composite drift: notebook {beta_comp_point!r} vs JSON {prim['composite_beta']['point']!r}"
)
assert np.isclose(se_comp, prim["composite_beta"]["se_hac"], atol=1e-9), "HAC SE drift"
assert np.isclose(t_comp, prim["composite_beta"]["t_stat"], atol=1e-9), "t-stat drift"
assert np.isclose(p_one_sided, prim["composite_beta"]["p_one_sided"], atol=1e-9), "p_one drift"
assert np.isclose(ci95_lower_one_sided, prim["composite_beta"]["ci95_lower_one_sided"],
                  atol=1e-9), "CI lower drift"
for col_name in ["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"]:
    assert np.isclose(individual[col_name]["point"],
                      prim["coefficients"][col_name]["point"], atol=1e-9), f"{col_name} point drift"
    assert np.isclose(individual[col_name]["se_hac"],
                      prim["coefficients"][col_name]["se_hac"], atol=1e-9), f"{col_name} se drift"
assert sign_pattern == prim["individual_lag_sign_pattern_b6_b9_b12"], "sign pattern drift"

# ── Off-spec dummied variant preview (used in §4 trio for full §4 report) ──
dummy_series = (df["timestamp_utc"] >= pd.Timestamp("2021-01-01", tz="UTC")).astype(int).to_numpy()
n_dummy_ones = int(dummy_series.sum())

# ── Persist NB02 §2 results to handoff dict (used by §3 + §4) ──────────────
nb02_primary = {
    "n": n,
    "newey_west_bandwidth": bw,
    "coefficients": {
        "const": {
            "point": float(fit.params[0]),
            "se_hac": float(fit.bse[0]),
            "t": float(fit.params[0] / fit.bse[0]),
            "p_two_sided": float(fit.pvalues[0]),
        },
        **individual,
    },
    "composite_beta": {
        "point": beta_comp_point,
        "se_hac": se_comp,
        "t_stat": t_comp,
        "p_one_sided": p_one_sided,
        "p_two_sided": p_two_sided,
        "ci95_lower_one_sided": ci95_lower_one_sided,
        "ci95_two_sided_lower": ci_two_lower,
        "ci95_two_sided_upper": ci_two_upper,
    },
    "individual_lag_sign_pattern_b6_b9_b12": sign_pattern,
    "rsquared": float(fit.rsquared),
    "rsquared_adj": float(fit.rsquared_adj),
    "joint_lags_F": float(f_test.fvalue),
    "joint_lags_p": float(f_test.pvalue),
    "diagnostics": {
        "breusch_pagan_lm": float(bp_lm),
        "breusch_pagan_p": float(bp_p),
        "durbin_watson": dw,
        "jarque_bera_stat": float(jb_stat),
        "jarque_bera_p": float(jb_p),
        "skew": float(skew_resid),
        "kurtosis_raw": float(kurt_resid_raw),
        "excess_kurtosis": excess_kurt,
        "adf_stat": float(adf_stat),
        "adf_p": float(adf_p),
        "adf_lag_used": int(adf_lag),
        "adf_n_used": int(adf_n),
        "ljung_box_lag4_stat": float(lb.loc[4, "lb_stat"]),
        "ljung_box_lag4_p": float(lb.loc[4, "lb_pvalue"]),
        "ljung_box_lag12_stat": float(lb.loc[12, "lb_stat"]),
        "ljung_box_lag12_p": float(lb.loc[12, "lb_pvalue"]),
    },
}

# ── Display ────────────────────────────────────────────────────────────────
print("=== §2 Primary OLS — spec §5.3 verbatim (HAC NW bw=4) ===")
print(fit.summary(xname=column_names_primary))

print("\n=== Composite β = β_6 + β_9 + β_12 (contrast c=[0,1,1,1]) ===")
print(f"  point         = {beta_comp_point:+.8e}")
print(f"  HAC SE        = {se_comp:.8e}")
print(f"  t-stat        = {t_comp:+.6f}")
print(f"  p one-sided   = {p_one_sided:.6e}")
print(f"  p two-sided   = {p_two_sided:.6e}")
print(f"  one-sided 95% CI lower = {ci95_lower_one_sided:+.8e}")
print(f"  two-sided 95% CI       = [{ci_two_lower:+.8e}, {ci_two_upper:+.8e}]")
print(f"  individual lag sign pattern (β_6/β_9/β_12): {sign_pattern}")
print(f"\nCommitted JSON sha256 OK: {committed_json_sha}")
print(f"Numerical reconciliation against primary_ols.json: PASS (atol=1e-9)")
print(f"\nResidual diagnostics: skew={skew_resid:+.4f}, excess_kurt={excess_kurt:+.4f}, "
      f"BP_p={bp_p:.4f}, DW={dw:.4f}, JB_p={jb_p:.4f}, ADF_p={adf_p:.4f}")
print(f"Ljung-Box L=4: stat={lb.loc[4, 'lb_stat']:.4f}, p={lb.loc[4, 'lb_pvalue']:.4e}")
print(f"Ljung-Box L=12: stat={lb.loc[12, 'lb_stat']:.4f}, p={lb.loc[12, 'lb_pvalue']:.4e}")
print(f"\nmarco2018_dummy ones (informational; off-spec only): {n_dummy_ones}")

=== §2 Primary OLS — spec §5.3 verbatim (HAC NW bw=4) ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.168
Model:                            OLS   Adj. R-squared:                  0.149
Method:                 Least Squares   F-statistic:                     10.52
Date:                Tue, 28 Apr 2026   Prob (F-statistic):           3.05e-06
Time:                        19:54:12   Log-Likelihood:                 184.50
No. Observations:                 134   AIC:                            -361.0
Df Residuals:                     130   BIC:                            -349.4
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

**Trio 2 — interpretation (numerical readout against committed `primary_ols.json`)**

**Individual lag coefficients (HAC NW bw=4):**

| Coef | Point | HAC SE | t | p (two-sided) | sign |
|---|---|---|---|---|---|
| const | −0.5161 | 0.1993 | −2.589 | 0.00963 | − |
| log_cop_usd_lag6 | **+0.10889931** | 0.07767 | +1.402 | 0.16090 | + |
| log_cop_usd_lag9 | **+0.01164843** | 0.08304 | +0.140 | 0.88845 | + |
| log_cop_usd_lag12 | **+0.01616210** | 0.08331 | +0.194 | 0.84618 | + |

**Composite-β contrast (load-bearing per spec §5.3):**

- Point: **+0.13670985** (β_6 + β_9 + β_12, contrast `c = [0, 1, 1, 1]`)
- HAC SE: **0.02465197** (composite-SE deflation explained below)
- t-stat: **+5.5456**
- One-sided p (H1: β > 0): **≈ 1.46×10⁻⁸** (rejection by ~5.5σ)
- One-sided 95% CI lower bound: **+0.09616** (excludes zero by ~3.9 SE)
- Two-sided 95% CI: [+0.0884, +0.1850]
- R² = 0.1677 (R²_adj = 0.1485); joint F-test β_6 = β_9 = β_12 = 0: F = 10.52, p = 3.05×10⁻⁶

**Lag pattern: +/+/+** — all three lags positive, consistent with the BPO Baumol → US-Colombia wage-arbitrage → US-client offshoring chain at the spec §5.4 6–12-month contracting-cycle horizon.

**Composite-vs-individual-lag SE structure (per RC FLAG #3 acknowledgement).** Individual-lag p-values inflate dramatically (β_9 p_two = 0.888, β_12 p_two = 0.846) because the three lagged COP/USD series are highly serially correlated — the same monthly TRM appears in three overlapping windows and the regressor cross-correlation inflates the diagonal HAC SEs. Composite SE deflates because the off-diagonal HAC entries are substantially negative (e.g. cov(β_6, β_9) ≈ −0.0028, cov(β_9, β_12) ≈ −0.0038): under the linear-restriction variance formula `Var(c'β) = c'Σc`, these negative covariances *subtract* from the composite variance, yielding `se_composite = 0.0247` rather than the naive `sqrt(0.0777² + 0.0830² + 0.0833²) = 0.139`. The composite test is therefore the *correct* application of the spec §5.3 pre-registered hypothesis on the same coefficient triple — it is not a different hypothesis, it is the right SE for the right hypothesis. Reading "individual lags weren't significant" is a methodological error.

**Lag-magnitude concentration acknowledgement (per RC FLAG #3 honest read).** β_6 / β_composite = 0.1089 / 0.1367 = 79.6%. The composite is concentrated at the lag-6 horizon, with lag-9 and lag-12 contributing the residual 20.4%. The honest narrative read is "concentrated at the early end of the BPO 6–12-month contracting-cycle window," NOT "uniformly spread across 6–12 months." This is a legitimate finding (consistent with the Beerepoot-Hendriks 2013 short-tail offshoring response) and is preserved as such in the §5 PRIMARY_RESULTS.md memo.

**Diagnostics readout:**

| Test | Stat | p | Read |
|---|---|---|---|
| Breusch-Pagan | 2.07 | 0.558 | No heteroskedasticity |
| Durbin-Watson | 1.416 | — | Mild +AR; HAC absorbs |
| Jarque-Bera | 0.87 | 0.648 | Residuals ≈ Gaussian |
| Skew | −0.0511 | — | `|skew| ≤ 1` → §3.4 B-ii does NOT fire |
| Excess kurtosis | +0.3811 | — | `≤ 3` → §3.4 B-ii does NOT fire |
| ADF on residuals | −3.80 | 0.00292 | Stationary (rejects unit root at 5%) |
| Ljung-Box L=4 | 23.53 | 9.91×10⁻⁵ | **Residual AR present** |
| Ljung-Box L=12 | 39.47 | 8.79×10⁻⁵ | **AR at annual lag** |

**Ljung-Box rejections at L=4 and L=12 are ANTICIPATED, not a deviation.** Spec §7 R4 pre-registered HAC NW with L=12 as the explicit robustness check for residual autocorrelation beyond the bandwidth-4 Newey-West kernel. The presence of residual AR is exactly what motivated the R4 specification at spec-authoring time, and NB03 §3 R4 will report whether the composite-β verdict survives the wider HAC kernel. This is a within-design observation — composite-β and its HAC-NW(4) SE are unbiased under the §5.3 pre-registration regardless of the residual AR; the SE is just sensitive to the bandwidth choice, which is precisely what R4 probes.

**Cross-check vs committed `primary_ols.json` (sha256 `d4790e743cdec62f1368cab1833e1266cb2da763d7c0931dd732bdf3d17938cf`):** all four composite-β scalars (`point`, `se_hac`, `t_stat`, `p_one_sided`) agree to 1e-9 absolute tolerance; all six individual-lag scalars agree to 1e-9; sign pattern matches exactly. **Round-trip PASS.** Byte-deterministic Option-β reproduction confirmed.

**Trio 2 PASS. NB02 §3 (verdict-tree mapping) may proceed.**

## §3 Verdict-tree mapping per spec §3.1 + §8.1

Trio 3: evaluate spec decision tree on observed (composite β, p_one_sided, sign); classify verdict ∈ {PASS, FAIL, ESCALATE, SUBSTRATE_TOO_NOISY}. Cross-check spec §3.1 PASS condition (`β > 0 AND p_one ≤ 0.05`); §3.3 Clause A non-firing (`p ∉ (0.05, 0.20]`); §3.4 Clause B / B-ii non-firing (`β > 0 AND |skew| ≤ 1 AND excess kurt ≤ 3`). Final verdict integration with NB03 R-consistency deferred to NB03 §5.

**Trio 3 — why-block (4-part citation block per `feedback_notebook_citation_block`)**

1. **Reference.** Spec [@abrigoSimpleBetaSpec2026] sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`. Sections governing this trio:
   - **§3.1 PASS-trigger.** `β_composite > 0 AND p_one_sided ≤ ALPHA_ONE_SIDED (= 0.05)`.
   - **§3.3 ESCALATE-trigger Clause A.** `β_composite > 0 AND p_one_sided ∈ (0.05, 0.20]` → run §5.5 escalation suite (quantile / GARCH-X / lower-tail) and record ESCALATE-PASS / ESCALATE-FAIL per §3.4.
   - **§3.4 ESCALATE-trigger Clause B and B-ii.** Clause B fires when `β > 0 AND p > 0.20` (or `β ≤ 0 AND p > 0.05`) IF both B-i (sign-stability) AND B-ii (`|skew| > 1` OR `excess_kurtosis > 3`) hold; otherwise the cell is FAIL per §3.2 + §8.1-4(c).
   - **§3.5 SUBSTRATE_TOO_NOISY.** Override fires when ≥ 3 of the four R-rows in NB03 sign-flip the composite-β relative to primary; this triggers a verdict override irrespective of the primary §3.1 trigger.
   - **§8.1 joint verdict integration (rules evaluated in order).** N gate → primary trigger evaluation → ESCALATE branch → R-consistency aggregation → final verdict.

2. **Why used (mechanical decision-tree application, no judgment).** The verdict tree is the spec-pinned anti-fishing primitive. Every threshold (`α = 0.05`, `ESCALATE_P_UPPER = 0.20`, `|skew| > 1`, `excess_kurtosis > 3`, R-row sign-flip count `≥ 3`) is fixed at spec-authoring time and immutable per §9.1; the spec sha256 is the cryptographic guarantee that the verdict that emerges from this trio is not the result of post-data threshold tuning. Mechanical evaluation here removes the temptation (and the appearance) of "judgement-call" verdicts. The function below evaluates each spec-pinned condition explicitly so the verdict's provenance is auditable line-by-line against the spec text.

3. **Relevance to results.** This trio produces the §8.1-4 *primary verdict primitive* — the input to the final joint verdict that NB03 §5 emits after R-consistency aggregation. If this trio returns PASS, the joint verdict is PASS unless NB03 R-consistency = DISAGREE or ≥3 sign-flips fire SUBSTRATE_TOO_NOISY (§3.5 override). If this trio returns ESCALATE, NB03 must run the full §5.5 escalation suite. If FAIL, the iteration closes per Phase-A.0 §11.A unless the convex-payoff escalation path opens.

4. **Connection to simulator (Stage-2 M-sketch gate).** The verdict here is the gate for Stage-2 M-sketch authoring per the framework section "Iteration order (default — target-population dominant)" in CLAUDE.md. PASS at this trio (combined with NB03 R-consistency = AGREE or MIXED) unblocks the ideal-scenario Panoptic-position M-sketch — the next exit criterion in the staged-correctness sequence. ESCALATE keeps the iteration alive but pushes the M-sketch decision to the convex-payoff escalation suite. FAIL closes the (Y, X) cell and informs the X-search prior for the next iteration of the framework.

In [3]:
"""§3 — verdict-tree mapping per spec §3.1 / §3.3 / §3.4 / §3.5 / §8.1.

Mechanical evaluation of the spec-pinned decision tree on the §2 composite-β
scalars. No judgment; every threshold is read from env (mirroring spec
constants per §9.1 immutability).

Identifier convention: Python 3 forbids the SECTION SIGN ('§', U+00A7) in
identifiers (PEP 3131; it is not in the Lu/Ll/Lt/Lm/Lo/Mn/Mc/Nd/Pc set).
This module names the spec sections with a leading 'sec' instead — the dict
keys returned to the JSON layer use the '§N.M' form, but Python identifiers
use 'secN_M'.
"""
from __future__ import annotations

from typing import Any


def evaluate_decision_tree(
    *,
    beta_comp: float,
    p_one: float,
    sign_pattern: str,
    skew: float,
    excess_kurt: float,
    alpha_one_sided: float = env.ALPHA_ONE_SIDED,
    escalate_p_upper: float = env.ESCALATE_P_UPPER,
) -> dict[str, Any]:
    """Apply spec §3.1 / §3.3 / §3.4 trigger primitives to the §2 results.

    Returns a dict with one boolean per spec-pinned condition + a verdict
    classification (PASS / ESCALATE / FAIL). The N gate (§3.6) and the
    SUBSTRATE_TOO_NOISY override (§3.5) are evaluated separately:
        - N gate is checked at NB02 §1 (panel handoff verify) — if it failed
          the notebook would have HALTed before reaching §3.
        - §3.5 SUBSTRATE_TOO_NOISY is an NB03 override (requires R-row
          sign-flip aggregation), so it cannot fire from NB02 alone.
    """
    # §3.1 PASS-trigger: β > 0 AND p_one ≤ α
    primary_pass_sec3_1 = (beta_comp > 0.0) and (p_one <= alpha_one_sided)

    # §3.3 ESCALATE-trigger Clause A: β > 0 AND p_one ∈ (α, ESCALATE_P_UPPER]
    clause_A_escalate_sec3_3 = (
        (beta_comp > 0.0)
        and (alpha_one_sided < p_one <= escalate_p_upper)
    )

    # §3.4 Clause B-ii: |skew| > 1 OR excess_kurtosis > 3
    clause_B_ii_kurt_skew_trigger = (abs(skew) > 1.0) or (excess_kurt > 3.0)

    # §3.4 Clause B FAIL branch (per §8.1-4(c) extension):
    # β ≤ 0 OR p > ESCALATE_P_UPPER (and Clause B-i AND B-ii do not both fire) → FAIL
    clause_B_fail_branch_sec3_4 = (
        (beta_comp <= 0.0)
        or (p_one > escalate_p_upper)
    )

    # Final verdict from primary alone (NB03 R-consistency overlay deferred)
    if primary_pass_sec3_1:
        verdict = "PASS"
    elif clause_A_escalate_sec3_3:
        verdict = "ESCALATE"  # Clause A
    elif (beta_comp > 0.0 or p_one > alpha_one_sided) and clause_B_ii_kurt_skew_trigger:
        # Clause B fires only if B-ii (kurtosis/skew) fires; B-i (sign-stability)
        # is an NB03-input check, so we treat it as "not-yet-falsified" here
        # and report ESCALATE pending NB03 confirmation.
        verdict = "ESCALATE"  # Clause B (pending NB03 B-i confirmation)
    else:
        verdict = "FAIL"

    return {
        "thresholds": {
            "alpha_one_sided": alpha_one_sided,
            "escalate_p_upper": escalate_p_upper,
            "skew_abs_upper_clause_B_ii": 1.0,
            "excess_kurtosis_upper_clause_B_ii": 3.0,
        },
        "observed": {
            "beta_composite": beta_comp,
            "p_one_sided": p_one,
            "sign_pattern_individual_lags": sign_pattern,
            "skew_residuals": skew,
            "excess_kurtosis_residuals": excess_kurt,
        },
        "checks": {
            "primary_pass_sec_3_1": primary_pass_sec3_1,
            "clause_A_escalate_sec_3_3_fires": clause_A_escalate_sec3_3,
            "clause_B_ii_kurt_skew_trigger": clause_B_ii_kurt_skew_trigger,
            "clause_B_fail_branch_sec_3_4": clause_B_fail_branch_sec3_4,
        },
        "verdict_primary_preliminary": verdict,
        "joint_verdict_note": (
            "Final joint verdict per spec §8.1 step 4 requires NB03 R-row "
            "consistency aggregation. If R-consistency ∈ {AGREE, MIXED}, "
            "primary PASS propagates to joint PASS. If R-consistency = "
            "DISAGREE OR ≥3 sign-flips → §3.5 SUBSTRATE_TOO_NOISY override "
            "= joint FAIL (irrespective of primary verdict)."
        ),
    }


# ── Apply to §2 observed scalars ───────────────────────────────────────────
verdict_payload = evaluate_decision_tree(
    beta_comp=nb02_primary["composite_beta"]["point"],
    p_one=nb02_primary["composite_beta"]["p_one_sided"],
    sign_pattern=nb02_primary["individual_lag_sign_pattern_b6_b9_b12"],
    skew=nb02_primary["diagnostics"]["skew"],
    excess_kurt=nb02_primary["diagnostics"]["excess_kurtosis"],
)

# ── Display ────────────────────────────────────────────────────────────────
print("=== §3 Verdict-tree mapping per spec §3.1 / §3.3 / §3.4 ===")
print("\nThresholds (spec-pinned, §9.1 immutable):")
for k, v in verdict_payload["thresholds"].items():
    print(f"  {k:45s} = {v}")
print("\nObserved (from §2):")
for k, v in verdict_payload["observed"].items():
    if isinstance(v, float):
        print(f"  {k:45s} = {v:+.8e}")
    else:
        print(f"  {k:45s} = {v}")
print("\nDecision-tree checks:")
for k, v in verdict_payload["checks"].items():
    print(f"  {k:45s} = {v}")
print(f"\nPrimary verdict (preliminary, pre-NB03 R-consistency): "
      f"{verdict_payload['verdict_primary_preliminary']}")
print(f"\nJoint-verdict note: {verdict_payload['joint_verdict_note']}")

# Cross-check: spec verdict from §2 round-trip JSON (sanity)
committed_primary_verdict = json.loads(env.PRIMARY_OLS_JSON.read_text())[
    "primary_spec_verbatim"]["composite_beta"]["verdict_alpha_05_one_sided"]
assert committed_primary_verdict == verdict_payload["verdict_primary_preliminary"], (
    f"Verdict drift: notebook {verdict_payload['verdict_primary_preliminary']!r} "
    f"vs committed JSON {committed_primary_verdict!r}"
)
print(f"\nVerdict cross-check vs committed primary_ols.json: PASS "
      f"(both = {committed_primary_verdict!r})")

=== §3 Verdict-tree mapping per spec §3.1 / §3.3 / §3.4 ===

Thresholds (spec-pinned, §9.1 immutable):
  alpha_one_sided                               = 0.05
  escalate_p_upper                              = 0.2
  skew_abs_upper_clause_B_ii                    = 1.0
  excess_kurtosis_upper_clause_B_ii             = 3.0

Observed (from §2):
  beta_composite                                = +1.36709847e-01
  p_one_sided                                   = +1.46478143e-08
  sign_pattern_individual_lags                  = +/+/+
  skew_residuals                                = -5.10919954e-02
  excess_kurtosis_residuals                     = +3.81103312e-01

Decision-tree checks:
  primary_pass_sec_3_1                          = True
  clause_A_escalate_sec_3_3_fires               = False
  clause_B_ii_kurt_skew_trigger                 = False
  clause_B_fail_branch_sec_3_4                  = False

Primary verdict (preliminary, pre-NB03 R-consistency): PASS

Joint-verdict note: Final joint

**Trio 3 — interpretation (mechanical verdict-tree readout)**

**Spec-pinned thresholds (§9.1 immutable, sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`):**

| Threshold | Value | Spec ref |
|---|---|---|
| `alpha_one_sided` | 0.05 | §3.1 |
| `escalate_p_upper` | 0.20 | §3.3 Clause A |
| Clause B-ii skew bound | `|skew| > 1` | §3.4 |
| Clause B-ii kurt bound | `excess_kurt > 3` | §3.4 |

**Observed (from §2):** β_composite = +0.13670985, p_one_sided = 1.46×10⁻⁸, sign pattern +/+/+, skew = −0.0511, excess kurtosis = +0.3811.

**Decision-tree checks:**

| Check | Truth value | Spec ref |
|---|---|---|
| `primary_pass_§3_1` (β > 0 AND p_one ≤ 0.05) | **TRUE** | §3.1 |
| `clause_A_escalate_§3_3_fires` (β > 0 AND p_one ∈ (0.05, 0.20]) | FALSE | §3.3 |
| `clause_B_ii_kurt_skew_trigger` (`|skew|>1` OR `excess_kurt>3`) | FALSE (`|−0.0511|=0.05 ≤ 1`; `0.38 ≤ 3`) | §3.4 B-ii |
| `clause_B_fail_branch_§3_4` (β ≤ 0 OR p > 0.20) | FALSE | §3.4 |

**Primary verdict (preliminary, pre-NB03 R-consistency): PASS.**

The §3.1 PASS-trigger fires unambiguously: β_composite is positive (+0.137 > 0), and p_one_sided (1.46×10⁻⁸) is ~6 orders of magnitude below the α = 0.05 threshold. The §3.3 Clause A ESCALATE branch does not fire (p_one is far below the lower bound of the ESCALATE band). The §3.4 Clause B-ii trigger does not fire (residuals are near-Gaussian per the Jarque-Bera p = 0.648 readout in §2). The §3.4 Clause B FAIL branch does not fire (β > 0, p ≤ 0.20).

**Cross-check vs committed `primary_ols.json` `verdict_alpha_05_one_sided` field:** notebook verdict matches committed verdict (both = `"PASS"`). Round-trip on the verdict primitive itself: PASS.

**Joint-verdict integration deferred to NB03 §5 per spec §8.1 step 4.** The final joint verdict requires the four-row R-consistency aggregation from NB03 (R1 = regime dummy; R2 = BPO-narrow J+M+N; R3 = raw-share OLS without logit; R4 = HAC NW L=12). Per §8.1-4(a):

- If NB03 R-consistency ∈ {AGREE, MIXED}: primary PASS propagates to joint PASS. Stage-2 M-sketch unblocked.
- If NB03 R-consistency = DISAGREE OR ≥ 3 R-rows sign-flip the composite-β relative to primary: §3.5 SUBSTRATE_TOO_NOISY override fires → joint FAIL (irrespective of primary §3.1 PASS).

The first contingency (R-consistency AGREE/MIXED → joint PASS) is the design's intended path; it would unblock the Stage-2 ideal-scenario Panoptic perpetual M-sketch on the Colombian-structural-transformation index. The second contingency (R-consistency DISAGREE → SUBSTRATE_TOO_NOISY override) is the design's defense against a primary-PASS that is fragile to the four single-dimension robustness perturbations enumerated in spec §7.

**No escalation triggered.** Spec §3.3 Clause A requires p ∈ (0.05, 0.20]; observed p = 1.46×10⁻⁸ is below that band, so the §5.5 escalation suite (quantile / GARCH-X / lower-tail) is not run from NB02 alone. (NB03 R4 will independently re-test the composite-β at HAC NW L=12 to address the Ljung-Box AR readouts from §2 diagnostics.)

**Trio 3 PASS. NB02 §4 (off-spec sensitivity / anti-fishing transparency) may proceed.**

## §4 Off-spec sensitivity — orchestrator-brief variant (NON-AUTHORITATIVE)

Trio 4 (anti-fishing transparency): the Phase-2 orchestrator brief specified a `marco2018_dummy` in primary; spec §5.3 verbatim has no such dummy in primary (it is the R1 robustness alternative ONLY per §6). Spec governs per §9.1 / §9.2 / §9.7 + memory `feedback_pathological_halt_anti_fishing_checkpoint`. Authoritative result = §2 + §3 above. The dummied variant is reported here as off-spec sensitivity ONLY; it would have mapped to ESCALATE Clause A had it been the spec primary.

Output: `estimates/PRIMARY_RESULTS.md` (§5).

**Trio 4 — why-block (4-part citation block per `feedback_notebook_citation_block`)**

1. **Reference.** Spec [@abrigoSimpleBetaSpec2026] sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`. Sections governing this trio:
   - **§9.1 (threshold immutability).** Once spec sha256 is pinned, no threshold (`α`, `ESCALATE_P_UPPER`, `MDES_SD`, etc.) and no specification choice (Y construction, X construction, lag set, primary contrast) may be re-tuned.
   - **§9.2 (Y-construction immutability).** The §5.1 Y_broad logit transform is the spec-pinned outcome; alternatives (raw share, BPO-narrow J+M+N) are R-row robustness rows in §7, not primary candidates.
   - **§9.7 (sha256 governance).** Spec changes require a new spec sha256 + CORRECTIONS block + 3-way review per §9.5. The committed-spec sha256 governs all downstream estimation per the cryptographic-pin chain.
   - **§5.3** (primary specification) — VERBATIM, no `marco2018_dummy` in primary; the dummy is the R1 robustness alternative ONLY per §6.
   - Memories `feedback_strict_tdd` (no untested code) and `feedback_pathological_halt_anti_fishing_checkpoint` (HALT + disposition memo + 3-way review for any post-data spec change) are operational invariants.

2. **Why used (orchestrator-brief vs spec-§5.3 contradiction transparency).** The Phase-2 orchestrator brief specified a "primary" OLS that includes a `marco2018_dummy` (1 if `t ≥ 2021-01-01` else 0). This contradicts spec §5.3 verbatim, which has NO such dummy in the primary. The contradiction was identified at Phase-2 dispatch (2026-04-28 PM). The spec sha256 was pinned at commit `21beffade` (2026-04-28 08:04 EDT) — 9.5 hours before the Phase-2 results emitted at 16:16–16:24 EDT. By spec §9.1 / §9.2 / §9.7 + the strict-TDD memory, the spec governs whenever it conflicts with downstream orchestration; the AR (Analytics Reporter) correctly executed the spec-verbatim primary in `task_2_1_primary_ols.py` and reported the dummied variant as off-spec sensitivity. This trio re-runs the dummied variant in NB02 with the same anti-fishing transparency framing — it is reported here ONLY so the reader can confirm what would have happened had the orchestrator-brief governed instead of the spec, and to make the spec's anti-fishing discipline auditable.

3. **Relevance to results.** The dummied variant produces a different composite-β, a different one-sided p, and would map to a different §3.3 verdict cell. It is NEVER allowed to override the §3 PASS verdict per spec §9.1; it exists in the audit trail solely as anti-fishing transparency. The §5 PRIMARY_RESULTS.md memo emitted by this trio explicitly tags the dummied variant as NON-AUTHORITATIVE and pins the spec-verbatim variant as the load-bearing result.

4. **Connection to simulator (governance, not Stage-2 design).** This trio has no direct simulator input. It is a meta-layer governance check that the Stage-2 M-sketch is built on the spec-pinned β and not on a post-data re-specified β. If the dummied β were the load-bearing β, the M-sketch would be parameterized on a different point estimate and a different SE — so the audit trail is essential for the integrity of the entire downstream Stage-2 → Stage-3 chain.

In [4]:
"""§4 — Off-spec sensitivity: orchestrator-brief variant WITH marco2018_dummy.

NON-AUTHORITATIVE. Per spec §9.1 / §9.2 / §9.7 the spec-verbatim primary
(NB02 §2 + §3) governs. This trio re-runs the dummied variant solely as
anti-fishing transparency and emits the §5 PRIMARY_RESULTS.md memo.
"""
from __future__ import annotations

# ── Re-build design matrix WITH marco2018_dummy (off-spec) ─────────────────
X_with_dummy = np.column_stack([X_design, dummy_series])
column_names_dummy = column_names_primary + ["marco2018_dummy"]

# ── Fit off-spec OLS with HAC-NW(4) ────────────────────────────────────────
fit_dummy = sm.OLS(y, X_with_dummy).fit(cov_type="HAC", cov_kwds={"maxlags": bw})

# Spec-§5.3-style composite β over the three lag betas only (dummy excluded)
c_dummy = np.array([0.0, 1.0, 1.0, 1.0, 0.0])
beta_hat_d = np.asarray(fit_dummy.params)
sigma_hac_d = np.asarray(fit_dummy.cov_params())
beta_comp_d = float(c_dummy @ beta_hat_d)
var_comp_d = float(c_dummy @ sigma_hac_d @ c_dummy)
assert var_comp_d > 0, f"Dummied variant: composite-β variance {var_comp_d} ≤ 0"
se_comp_d = float(np.sqrt(var_comp_d))
t_comp_d = beta_comp_d / se_comp_d
if t_comp_d > 0:
    p_one_d = float(1.0 - stats.norm.cdf(t_comp_d))
else:
    p_one_d = float(stats.norm.cdf(t_comp_d))
ci_one_lower_d = beta_comp_d - z_95 * se_comp_d

# Off-spec lag-sign pattern
individual_d = {}
for i, col in enumerate(["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"], start=1):
    b = float(fit_dummy.params[i])
    sign = "+" if b > 0 else ("-" if b < 0 else "0")
    individual_d[col] = {"point": b, "se_hac": float(fit_dummy.bse[i]), "sign": sign}
sign_pattern_d = "/".join(individual_d[c]["sign"] for c in
                          ["log_cop_usd_lag6", "log_cop_usd_lag9", "log_cop_usd_lag12"])

# Verdict the dummied variant WOULD have mapped to (NON-AUTHORITATIVE)
if beta_comp_d > 0 and p_one_d <= env.ALPHA_ONE_SIDED:
    dummied_verdict_would_be = "PASS"
elif beta_comp_d > 0 and env.ALPHA_ONE_SIDED < p_one_d <= env.ESCALATE_P_UPPER:
    dummied_verdict_would_be = "ESCALATE (Clause A)"
else:
    dummied_verdict_would_be = "FAIL"

# ── Cross-check vs committed off-spec block ────────────────────────────────
off_spec_committed = json.loads(env.PRIMARY_OLS_JSON.read_text())[
    "off_spec_sensitivity_orchestrator_brief"]
assert np.isclose(beta_comp_d, off_spec_committed["composite_beta"]["point"], atol=1e-9), (
    f"Off-spec composite drift: notebook {beta_comp_d!r} vs JSON "
    f"{off_spec_committed['composite_beta']['point']!r}"
)
assert np.isclose(se_comp_d, off_spec_committed["composite_beta"]["se_hac"], atol=1e-9)
assert np.isclose(t_comp_d, off_spec_committed["composite_beta"]["t_stat"], atol=1e-9)
assert np.isclose(p_one_d, off_spec_committed["composite_beta"]["p_one_sided"], atol=1e-9)

print("=== §4 Off-spec sensitivity (orchestrator-brief variant — NON-AUTHORITATIVE) ===")
print(f"\nComposite β over [const, lag6, lag9, lag12, marco2018_dummy] (c=[0,1,1,1,0]):")
print(f"  point        = {beta_comp_d:+.6e}")
print(f"  HAC SE       = {se_comp_d:.6e}")
print(f"  t-stat       = {t_comp_d:+.4f}")
print(f"  p one-sided  = {p_one_d:.6f}")
print(f"  one-sided 95% CI lower = {ci_one_lower_d:+.6e}")
print(f"  individual lag sign pattern (β_6/β_9/β_12): {sign_pattern_d}")
print(f"  marco2018_dummy point  = {float(fit_dummy.params[4]):+.6f} "
      f"(HAC SE {float(fit_dummy.bse[4]):.6f})")
print(f"\nVerdict the dummied variant WOULD map to (NON-AUTHORITATIVE): "
      f"{dummied_verdict_would_be}")
print(f"\nCross-check vs committed off-spec block in primary_ols.json: PASS (atol=1e-9)")

# ── Emit §5 PRIMARY_RESULTS.md memo ────────────────────────────────────────
env.ESTIMATES_DIR.mkdir(parents=True, exist_ok=True)

primary_results_md = f"""# Pair D NB02 — PRIMARY_RESULTS

**Spec sha256 (governing):** `{env.SPEC_SHA256}`
**Panel sha256:** `{env.PANEL_SHA256}`
**Committed primary_ols.json sha256:** `{env.PRIMARY_OLS_SHA256}`
**N:** {nb02_primary['n']} | **Window:** {env.WINDOW_START} → {env.WINDOW_END} | **HAC NW bw:** {nb02_primary['newey_west_bandwidth']}

## §1 PRIMARY (spec §5.3 verbatim — AUTHORITATIVE)

`Y_broad_logit = α + β_6·log_cop_usd_lag6 + β_9·log_cop_usd_lag9 + β_12·log_cop_usd_lag12 + ε`

| Coef | Point | HAC SE | t | p (two-sided) |
|---|---|---|---|---|
| const | {nb02_primary['coefficients']['const']['point']:+.6f} | {nb02_primary['coefficients']['const']['se_hac']:.6f} | {nb02_primary['coefficients']['const']['t']:+.3f} | {nb02_primary['coefficients']['const']['p_two_sided']:.4f} |
| log_cop_usd_lag6 | **{nb02_primary['coefficients']['log_cop_usd_lag6']['point']:+.6f}** | {nb02_primary['coefficients']['log_cop_usd_lag6']['se_hac']:.6f} | {nb02_primary['coefficients']['log_cop_usd_lag6']['t']:+.3f} | {nb02_primary['coefficients']['log_cop_usd_lag6']['p_two_sided']:.4f} |
| log_cop_usd_lag9 | **{nb02_primary['coefficients']['log_cop_usd_lag9']['point']:+.6f}** | {nb02_primary['coefficients']['log_cop_usd_lag9']['se_hac']:.6f} | {nb02_primary['coefficients']['log_cop_usd_lag9']['t']:+.3f} | {nb02_primary['coefficients']['log_cop_usd_lag9']['p_two_sided']:.4f} |
| log_cop_usd_lag12 | **{nb02_primary['coefficients']['log_cop_usd_lag12']['point']:+.6f}** | {nb02_primary['coefficients']['log_cop_usd_lag12']['se_hac']:.6f} | {nb02_primary['coefficients']['log_cop_usd_lag12']['t']:+.3f} | {nb02_primary['coefficients']['log_cop_usd_lag12']['p_two_sided']:.4f} |

**Composite β = β_6 + β_9 + β_12** (contrast `c = [0, 1, 1, 1]`, `Var(c'β) = c'Σ_HAC c`):

- Point: **{nb02_primary['composite_beta']['point']:+.8f}**
- HAC SE: **{nb02_primary['composite_beta']['se_hac']:.8f}**
- t-stat: **{nb02_primary['composite_beta']['t_stat']:+.4f}**
- One-sided p (H1: β > 0): **{nb02_primary['composite_beta']['p_one_sided']:.6e}**
- One-sided 95% CI lower: **{nb02_primary['composite_beta']['ci95_lower_one_sided']:+.6f}**
- Two-sided 95% CI: [{nb02_primary['composite_beta']['ci95_two_sided_lower']:+.6f}, {nb02_primary['composite_beta']['ci95_two_sided_upper']:+.6f}]
- Individual-lag sign pattern (β_6 / β_9 / β_12): **{nb02_primary['individual_lag_sign_pattern_b6_b9_b12']}**
- R² = {nb02_primary['rsquared']:.4f} (R²_adj = {nb02_primary['rsquared_adj']:.4f}); joint F-test β_6 = β_9 = β_12 = 0: F = {nb02_primary['joint_lags_F']:.4f}, p = {nb02_primary['joint_lags_p']:.6e}

### §2 §3 Verdict-tree mapping (preliminary, pre-NB03 R-consistency)

| Spec ref | Check | Truth |
|---|---|---|
| §3.1 | `β > 0 AND p_one ≤ 0.05` | **{verdict_payload['checks']['primary_pass_sec_3_1']}** |
| §3.3 Clause A | `β > 0 AND p_one ∈ (0.05, 0.20]` | {verdict_payload['checks']['clause_A_escalate_sec_3_3_fires']} |
| §3.4 Clause B-ii | `\\|skew\\|>1` OR `excess_kurt>3` | {verdict_payload['checks']['clause_B_ii_kurt_skew_trigger']} |
| §3.4 Clause B FAIL | `β ≤ 0` OR `p > 0.20` | {verdict_payload['checks']['clause_B_fail_branch_sec_3_4']} |

**PRIMARY VERDICT (preliminary): {verdict_payload['verdict_primary_preliminary']}**

Joint-verdict integration deferred to NB03 §5 per spec §8.1 step 4.

### §3 Diagnostics (residuals from primary OLS)

- Breusch-Pagan: LM = {nb02_primary['diagnostics']['breusch_pagan_lm']:.4f}, p = {nb02_primary['diagnostics']['breusch_pagan_p']:.4f} → no heteroskedasticity
- Durbin-Watson: {nb02_primary['diagnostics']['durbin_watson']:.4f} → mild +AR; HAC absorbs
- Jarque-Bera: stat = {nb02_primary['diagnostics']['jarque_bera_stat']:.4f}, p = {nb02_primary['diagnostics']['jarque_bera_p']:.4f} → residuals near-Gaussian
- Skew: {nb02_primary['diagnostics']['skew']:+.4f} (`|skew| ≤ 1` → §3.4 B-ii does NOT fire)
- Excess kurtosis: {nb02_primary['diagnostics']['excess_kurtosis']:+.4f} (`≤ 3` → §3.4 B-ii does NOT fire)
- ADF on residuals: stat = {nb02_primary['diagnostics']['adf_stat']:.4f}, p = {nb02_primary['diagnostics']['adf_p']:.4f} → stationary
- Ljung-Box L=4: stat = {nb02_primary['diagnostics']['ljung_box_lag4_stat']:.4f}, p = {nb02_primary['diagnostics']['ljung_box_lag4_p']:.4e} → AR present (ANTICIPATED; addressed by NB03 R4 HAC NW L=12)
- Ljung-Box L=12: stat = {nb02_primary['diagnostics']['ljung_box_lag12_stat']:.4f}, p = {nb02_primary['diagnostics']['ljung_box_lag12_p']:.4e} → AR at annual lag (ANTICIPATED; addressed by NB03 R4 HAC NW L=12)

## §4 OFF-SPEC SENSITIVITY (orchestrator-brief variant WITH marco2018_dummy — NON-AUTHORITATIVE)

The Phase-2 orchestrator brief specified a "primary" OLS that adds `marco2018_dummy` (1 if `t ≥ 2021-01-01` else 0). This contradicts spec §5.3 verbatim, which has NO such dummy in the primary; per §6 the dummy is the R1 robustness alternative ONLY. Per §9.1 / §9.2 / §9.7 the spec governs. The dummied variant is reported here for transparency.

**Composite β over [const, lag6, lag9, lag12, marco2018_dummy]** (`c = [0, 1, 1, 1, 0]`):

- Point: **{beta_comp_d:+.6f}**
- HAC SE: {se_comp_d:.6f}
- t-stat: {t_comp_d:+.4f}
- One-sided p: **{p_one_d:.6f}**
- One-sided 95% CI lower: {ci_one_lower_d:+.6f}
- Individual-lag sign pattern: **{sign_pattern_d}** (lag-12 flips sign)
- `marco2018_dummy` point: {float(fit_dummy.params[4]):+.6f} (HAC SE {float(fit_dummy.bse[4]):.6f})

**Verdict the dummied variant WOULD have mapped to (NON-AUTHORITATIVE): {dummied_verdict_would_be}.** This would have triggered the §3.3 Clause A escalation suite per spec §5.5 — quantile / GARCH-X / lower-tail. Per spec §9.1, the spec governs and the §3 verdict (PASS) stands.

## §5 Cryptographic round-trip

- `primary_ols.json` sha256: `{env.PRIMARY_OLS_SHA256}` — verified
- All §2 composite-β scalars reproduced to 1e-9 absolute tolerance against committed JSON
- All §4 off-spec composite-β scalars reproduced to 1e-9 absolute tolerance against committed JSON `off_spec_sensitivity_orchestrator_brief` block

**Recommendation for NB03 (Phase 2.2 / 2.3).** Spec-verbatim primary triggers §3.1 PASS. Per §8.1 step 4(a), if R-consistency (R1–R4 from NB03) is AGREE or MIXED, joint verdict = PASS and Stage-2 M sketch unblocked. Escalation NOT triggered. Task 2.2 must run; residual-AR diagnostic pre-validates R4 (HAC L=12) materiality.
"""

env.PRIMARY_RESULTS_PATH.write_text(primary_results_md)
primary_results_sha = hashlib.sha256(env.PRIMARY_RESULTS_PATH.read_bytes()).hexdigest()
print(f"\nWrote PRIMARY_RESULTS.md: {env.PRIMARY_RESULTS_PATH}")
print(f"  sha256: {primary_results_sha}")

=== §4 Off-spec sensitivity (orchestrator-brief variant — NON-AUTHORITATIVE) ===

Composite β over [const, lag6, lag9, lag12, marco2018_dummy] (c=[0,1,1,1,0]):
  point        = +8.145583e-02
  HAC SE       = 5.805108e-02
  t-stat       = +1.4032
  p one-sided  = 0.080282
  one-sided 95% CI lower = -1.402970e-02
  individual lag sign pattern (β_6/β_9/β_12): +/+/-
  marco2018_dummy point  = +0.028285 (HAC SE 0.026642)

Verdict the dummied variant WOULD map to (NON-AUTHORITATIVE): ESCALATE (Clause A)

Cross-check vs committed off-spec block in primary_ols.json: PASS (atol=1e-9)

Wrote PRIMARY_RESULTS.md: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/simple-beta-pair-d/notebooks/estimates/PRIMARY_RESULTS.md
  sha256: b565b198f06d8222b744a40666731c2380d1367e844b3a7c60eb6e11d883f437


**Trio 4 — interpretation (off-spec sensitivity readout, anti-fishing transparency)**

**Off-spec dummied composite-β readout:**

| Quantity | Value |
|---|---|
| Composite β (lag6 + lag9 + lag12) | **+0.0815** |
| HAC SE | 0.0581 |
| t-stat | +1.40 |
| One-sided p (H1: β > 0) | **0.0803** |
| One-sided 95% CI lower | −0.0140 |
| Individual lag sign pattern | **+/+/−** (lag-12 flips sign) |
| `marco2018_dummy` point | +0.0283 (HAC SE 0.0266) |

**Verdict the dummied variant WOULD have mapped to (NON-AUTHORITATIVE):** The composite p_one ≈ 0.080 falls in the §3.3 Clause A ESCALATE band `(0.05, 0.20]`. Had the orchestrator-brief governed instead of the spec, the verdict would be **ESCALATE (Clause A)** — triggering the §5.5 escalation suite (quantile regression on Y, GARCH-X, lower-tail / EVT analysis).

**Per spec §9.1, the spec governs.** The §3 verdict (PASS) stands. The dummied variant is reported here ONLY as anti-fishing transparency; it does NOT and CANNOT override the spec-pinned primary. Reading the dummied variant as "the actual result" would constitute a post-data spec-change without the §9.5 / §9.7 governance trail (CORRECTIONS block + 3-way review + new spec sha256), and is anti-fishing-banned.

**Why the dummied variant's composite is materially different.** The `marco2018_dummy` absorbs ~55% of the composite-β magnitude (committed `task_2_1_findings.md` §5 numerical breakdown). This is standard collinearity-induced obscuring: the methodology-break dummy is itself partially correlated with the lag-12 regressor (the 12-month lag straddles the Marco-2005 → Marco-2018 transition for the 2022 onward observations), so adding the dummy without theoretical pre-justification absorbs variance that the lag-12 coefficient would otherwise carry. This is precisely why the spec § 6 reserved the dummy as R1 robustness ONLY: applied alongside the empalme correction it would double-count the methodology-break adjustment, and applied without the empalme it absorbs cumulative-lag variance. R1 (NB03) will report the dummied variant on the same composite-contrast and aggregate it under the R-consistency rule of §7.2.

**Cross-check vs committed `primary_ols.json` `off_spec_sensitivity_orchestrator_brief` block:** all four off-spec composite-β scalars (`point`, `se_hac`, `t_stat`, `p_one_sided`) reproduce to 1e-9 absolute tolerance. Round-trip on the off-spec block: PASS.

**`PRIMARY_RESULTS.md` emitted to `estimates/PRIMARY_RESULTS.md`** with the spec-verbatim primary tagged AUTHORITATIVE and the dummied variant tagged NON-AUTHORITATIVE; sha256 printed in the code cell output for downstream pin-trail.

**Trio 4 PASS. NB02 complete.** Hand off to NB03 for R1–R4 robustness rows, R-consistency aggregation, and joint-verdict emission.